# Week 2.1 Comparing the sorting performances under different keep_ratio
## 1.Work flow Continue

In [1]:
#2026-5-16
# This block is to import the result and workflow last time.
import numpy as np
import pandas as pd
from scipy.fftpack import dct, idct
from pathlib import Path
import matplotlib.pyplot as plt
from kilosort.io import load_ops

SAVE_PATH = Path('F:\\ACADEMIC') / '.test_data' / 'ZFM-02370_mini.imec0.ap.short.bin'

n_chan = 385          # channel number
N = 267 # Neuron Number
dtype = 'int16'

# Load the saved Sorting results from the local drive.
results_dir = SAVE_PATH.parent / 'kilosort4'
ops = load_ops(results_dir / 'ops.npy')
camps = pd.read_csv(results_dir / 'cluster_Amplitude.tsv', sep='\t')['Amplitude'].values
contam_pct = pd.read_csv(results_dir / 'cluster_ContamPct.tsv', sep='\t')['ContamPct'].values
chan_map =  np.load(results_dir / 'channel_map.npy')
templates =  np.load(results_dir / 'templates.npy')
chan_best = (templates**2).sum(axis=1).argmax(axis=-1)
chan_best = chan_map[chan_best]
amplitudes = np.load(results_dir / 'amplitudes.npy')
st = np.load(results_dir / 'spike_times.npy')
clu = np.load(results_dir / 'spike_clusters.npy')

fs = ops['fs']
firing_rates = np.unique(clu, return_counts=True)[1] * fs / st.max()
dshift = ops['dshift']

In [2]:
whitened_data = np.load('whitened_data.npy')  # shape: (n_samples, n_channels)
n_samples, n_channels = whitened_data.shape
print(f"Shape: {n_samples} Samples × {n_channels} Channels")

Shape: 1350000 Samples × 385 Channels


In [3]:
import numpy as np
import torch
import os
from pathlib import Path
from scipy import signal
from scipy.spatial.distance import cdist
from scipy.fft import dct, idct
import kilosort
import warnings
warnings.filterwarnings('ignore')

## 2.Establish a reference cluster (uncompressed path)

In [4]:
baseline_res = np.load('baseline_results.npy', allow_pickle=True).item()

reference_templates = {}      
reference_spike_times = {}    
reference_spike_amps = {}     

clu = baseline_res.get('clu')
if clu is None and 'st0' in baseline_res:
    st0 = baseline_res['st0']
    if st0.ndim > 1:
        clu = st0[:, 1].astype(np.int32)

templates = baseline_res['templates']  # shape: (n_templates, n_samples_per_template, n_channels)

unique_clusters = np.unique(clu)
n_templates = templates.shape[0]

print(f"The number of reference cluster: {len(unique_clusters)}")
print(f"Template Shape: {templates.shape}")

for cluster_id in unique_clusters:
    spike_idx = clu == cluster_id
    reference_templates[cluster_id] = templates[cluster_id]
    
    # get spike time
    if 'st0' in baseline_res:
        st = baseline_res['st0']
        if st.ndim > 1:
            reference_spike_times[cluster_id] = st[spike_idx, 0]
        else:
            reference_spike_times[cluster_id] = st[spike_idx]
    
    # get spike am
    if 'amp' in baseline_res:
        reference_spike_amps[cluster_id] = baseline_res['amp'][spike_idx]

print(f"Reference_templates shape: {reference_templates[unique_clusters[0]].shape}")


The number of reference cluster: 267
Template Shape: (267, 61, 383)
Reference_templates shape: (61, 383)


In [5]:
def compress_decompress_dct(data, keep_ratio):
    n_samples, n_chan = data.shape
    recon_data = np.zeros_like(data, dtype=np.float32)
    
    for ch in range(n_chan):
        ch_signal = data[:, ch]
        dct_coeffs = dct(ch_signal, norm='ortho')
        n_keep = int(n_samples * keep_ratio)
        compressed_coeffs = np.zeros(n_samples, dtype=np.float32)
        compressed_coeffs[:n_keep] = dct_coeffs[:n_keep]
        recon_data[:, ch] = idct(compressed_coeffs, norm='ortho')
        
        if (ch + 1) % 100 == 0:
            print(f"  Processing Channel {ch+1}/{n_chan}...")
    
    return recon_data


def process_compressed_path(whitened_data, keep_ratio, cache_dir='./compressed_cache'):
    """
    Compression path: Compression → Decompression → Return to reconstructed data
    """
    os.makedirs(cache_dir, exist_ok=True)
    cache_file = os.path.join(cache_dir, f'reconstructed_ratio_{keep_ratio:.2f}.npy')
    
    if os.path.exists(cache_file):
        print(f"  Loaded from cache: {cache_file}")
        return np.load(cache_file)
    
    print(f"  Keep Ratio {keep_ratio*100:.0f}%: DCT Compressing...")
    recon_data = compress_decompress_dct(whitened_data, keep_ratio)
    
    print(f"  Saved to cache: {cache_file}")
    np.save(cache_file, recon_data)
    
    return recon_data

In [ ]:
def process_whitened_to_bin(whitened_data, keep_ratio):
    """The whitening data is compressed and saved as a bin file."""
    n_samples, n_chan = whitened_data.shape
    output_path = f'whitened_recon_ratio_{keep_ratio:.2f}.bin'
    
    if os.path.exists(output_path):
        expected_size = n_samples * n_chan * 4
        if os.path.getsize(output_path) == expected_size:
            print(f"  File {output_path} already exists and is the correct size; skip the compression step.")
            return output_path
        else:
            print(f"  The file {output_path} exists but may be corrupted; please recompress it...")
    
    print(f"  File created: {output_path}...")
    recon_data = np.memmap(output_path, dtype='float32', mode='w+', shape=(n_samples, n_chan))
    print(f"  DCT Compressing (Keep ratio={keep_ratio})...")
    
    for ch in range(n_chan):
        ch_signal = whitened_data[:, ch]
        dct_coeffs = dct(ch_signal, norm='ortho')
        n_keep = int(n_samples * keep_ratio)
        compressed_coeffs = np.zeros(n_samples, dtype='float32')
        compressed_coeffs[:n_keep] = dct_coeffs[:n_keep]
        recon_data[:, ch] = idct(compressed_coeffs, norm='ortho')
        
        if (ch + 1) % 100 == 0:
            print(f"  处理通道 {ch+1}/{n_chan}")
    
    recon_data.flush()
    del recon_data
    print(f"  压缩完成: {output_path}")
    return output_path


def process_whitened_to_kilo(bin_file, ratio, baseline_res, whitened_data_shape):
    """在压缩数据上使用原始模板匹配尖峰（保持原有功能）"""
    print(f"\n  --- 处理压缩数据 (keep_ratio={ratio}) ---")
    
    original_templates = baseline_res['templates']
    n_templates = original_templates.shape[0]
    n_samples, n_channels = whitened_data_shape
    
    print(f"  加载压缩数据: {bin_file}")
    compressed_data = np.memmap(bin_file, dtype='float32', mode='r', shape=(n_samples, n_channels))
    print(f"  数据: {n_samples} 样本, {n_channels} 通道")
    print(f"  使用 {n_templates} 个原始模板作为锚点")
    
    # 为每个模板找到最佳通道
    best_channels = []
    for temp_idx in range(n_templates):
        template = original_templates[temp_idx]
        channel_energy = np.sum(template ** 2, axis=0)
        best_channel = np.argmax(channel_energy)
        best_channels.append(best_channel)
    
    print("  🔍 使用互相关匹配尖峰...")
    st_list = []
    
    for temp_idx in range(n_templates):
        if (temp_idx + 1) % 50 == 0:
            print(f"    处理模板 {temp_idx+1}/{n_templates}")
        
        template = original_templates[temp_idx]
        best_ch = best_channels[temp_idx]
        template_waveform = template[:, best_ch]
        
        data_ch = compressed_data[:, best_ch]
        correlation = signal.correlate(data_ch, template_waveform, mode='same')
        threshold = 3.5 * np.std(correlation)
        peaks, _ = signal.find_peaks(correlation, height=threshold, distance=20)
        
        for peak in peaks:
            st_list.append([peak, temp_idx, correlation[peak]])
    
    st = np.array(st_list) if st_list else np.empty((0, 3))
    
    if len(st) == 0:
        print("  ✗ 未匹配到尖峰")
        return None
    
    print(f"  ✓ 匹配到 {len(st)} 个尖峰")
    
    rez_comp = {
        'st': st,
        'templates': original_templates,
        'clu': st[:, 1].astype(np.int32),
        'keep_ratio': ratio,
    }
    
    unique_clusters = np.unique(rez_comp['clu'])
    print(f"  📊 统计:")
    print(f"    - 活跃簇: {len(unique_clusters)} / {n_templates}")
    print(f"    - 总尖峰数: {len(st)}")
    print(f"    - 每活跃簇尖峰数: {len(st) / len(unique_clusters):.1f}")
    
    return rez_comp


In [6]:
def run_kilosort4_workflow(data, probe_file, results_name='temp_results', base_path=None):
    """
    使用你已有的工作流运行 Kilosort4
    
    这个方法会：
    1. 保存数据到临时二进制文件
    2. 创建结果目录
    3. 运行 kilosort 主流程
    4. 加载并返回结果
    """
    from kilosort import run_kilosort
    
    # 设置工作目录
    if base_path is None:
        base_path = Path('F:\\ACADEMIC') / '.test_data'
    else:
        base_path = Path(base_path)
    
    results_dir = base_path / results_name
    results_dir.mkdir(parents=True, exist_ok=True)
    
    # 保存数据
    binary_file = results_dir / 'data.bin'
    print(f"  保存数据到: {binary_file}")
    
    # 转换为 int16 并保存
    if data.dtype != np.int16:
        data_int16 = (data * 100).astype(np.int16)
    else:
        data_int16 = data
    data_int16.tofile(binary_file)
    
    # 构建配置
    ops = {
        'data_dir': str(results_dir),
        'binary_file': str(binary_file),
        'probe_file': probe_file,
        'fs': 30000,
        'nt0': 61,
        'Nt': 61,
        'n_chan_bin': data.shape[1],
        'dtype': 'int16',
    }
    
    # 运行 Kilosort4
    print("  运行 Kilosort4...")
    ops = run_kilosort(ops)
    
    # 加载结果
    from kilosort.io import load_ops
    
    ops = load_ops(results_dir / 'ops.npy')
    templates = np.load(results_dir / 'templates.npy')
    st = np.load(results_dir / 'spike_times.npy')
    clu = np.load(results_dir / 'spike_clusters.npy')
    
    # 构建结果字典
    rez = {
        'ops': ops,
        'templates': templates,
        'st': np.column_stack([st, np.zeros_like(st)]),  # 需要临时模板 ID
        'clu': clu,
        'xc': ops.get('xc'),
        'yc': ops.get('yc'),
        'fs': ops['fs'],
        'save_path': results_dir,
    }
    
    return rez

In [7]:
def run_compression_experiment(
    whitened_data,          # 白化后的原始数据
    baseline_rez,           # 你已有的 baseline 结果
    probe_file,             # 探针文件路径
    ratios=[0.1, 0.2, 0.3, 0.5],
    match_threshold=0.7,
    use_temp_dir=True       # 是否使用临时目录
):
    """
    完整的压缩实验流程
    
    Args:
        whitened_data: 白化后的数据 (n_samples, n_channels)
        baseline_rez: 未压缩路径的 Kilosort4 结果
        probe_file: 探针文件路径
        ratios: 压缩率列表
        match_threshold: 模板匹配阈值
        use_temp_dir: 是否使用临时目录（True）还是固定目录（False）
    """
    
    # 从 baseline 提取参考模板和统计信息
    reference_templates = {}
    reference_counts = {}
    
    # baseline 中的 clu 和 templates
    clu_ref = baseline_rez['clu']
    templates_ref = baseline_rez['templates']
    
    unique_clusters = np.unique(clu_ref)
    print(f"参考簇数量: {len(unique_clusters)}")
    
    for cluster_id in unique_clusters:
        if cluster_id >= 0 and cluster_id < templates_ref.shape[0]:
            reference_templates[cluster_id] = templates_ref[cluster_id]
            reference_counts[cluster_id] = np.sum(clu_ref == cluster_id)
    
    total_ref_spikes = sum(reference_counts.values())
    print(f"总参考尖峰数: {total_ref_spikes}")
    
    results = []
    
    for ratio in ratios:
        print("\n" + "="*60)
        print(f"处理压缩率: {ratio*100:.0f}%")
        print("="*60)
        
        # Step 1: 压缩并解压缩
        print("步骤 1: DCT 压缩 → 解压缩")
        n_samples, n_chan = whitened_data.shape
        reconstructed_data = np.zeros_like(whitened_data)
        
        for ch in range(n_chan):
            if ch % 100 == 0:
                print(f"  处理通道 {ch}/{n_chan}")
            ch_signal = whitened_data[:, ch]
            dct_coeffs = dct(ch_signal, norm='ortho')
            n_keep = int(n_samples * ratio)
            dct_coeffs[n_keep:] = 0
            reconstructed_data[:, ch] = idct(dct_coeffs, norm='ortho')
        
        # Step 2: 在重建数据上运行 Kilosort4
        print("步骤 2: 运行 Kilosort4")
        
        if use_temp_dir:
            # 使用临时目录
            comp_rez = run_kilosort4_on_data(
                data=reconstructed_data,
                probe_file=probe_file,
                params={'fs': 30000, 'nt0': 61}
            )
        else:
            # 使用固定目录，便于调试
            results_name = f'compressed_results_ratio_{int(ratio*100)}'
            comp_rez = run_kilosort4_workflow(
                data=reconstructed_data,
                probe_file=probe_file,
                results_name=results_name
            )
        
        # Step 3: 提取压缩结果中的模板和簇信息
        comp_clu = comp_rez['clu']
        comp_templates = comp_rez['templates']
        comp_st = comp_rez['st']
        
        # Step 4: 匹配尖峰到参考模板
        print("步骤 3: 匹配尖峰到参考模板")
        
        # 预展平参考模板
        ref_ids = list(reference_templates.keys())
        ref_vectors = []
        for ref_id in ref_ids:
            ref_vec = reference_templates[ref_id].flatten()
            ref_norm = ref_vec / (np.linalg.norm(ref_vec) + 1e-8)
            ref_vectors.append(ref_norm)
        ref_vectors = np.array(ref_vectors)
        
        assignments = []
        confidences = []
        
        for spike_idx, cluster_id in enumerate(comp_clu):
            if cluster_id < 0 or cluster_id >= comp_templates.shape[0]:
                assignments.append(-1)
                confidences.append(0)
                continue
            
            spike_template = comp_templates[cluster_id]
            spike_vec = spike_template.flatten()
            spike_norm = spike_vec / (np.linalg.norm(spike_vec) + 1e-8)
            
            similarities = np.dot(ref_vectors, spike_norm)
            best_idx = np.argmax(similarities)
            best_sim = similarities[best_idx]
            
            if best_sim > match_threshold:
                assignments.append(ref_ids[best_idx])
                confidences.append(best_sim)
            else:
                assignments.append(-1)
                confidences.append(0)
        
        assignments = np.array(assignments)
        
        # Step 5: 计算指标
        print("步骤 4: 计算指标")
        
        n_comp_spikes = len(comp_clu)
        n_matched = np.sum(assignments != -1)
        n_unmatched = n_comp_spikes - n_matched
        
        stability = n_matched / total_ref_spikes if total_ref_spikes > 0 else 0
        dropout_rate = 1 - stability
        novelty_rate = n_unmatched / n_comp_spikes if n_comp_spikes > 0 else 0
        
        # 每个神经元的灵敏度
        per_neuron_sensitivity = {}
        for ref_id, expected in reference_counts.items():
            found = np.sum(assignments == ref_id)
            per_neuron_sensitivity[ref_id] = found / expected if expected > 0 else 0
        
        metrics = {
            'stability': stability,
            'dropout_rate': dropout_rate,
            'novelty_rate': novelty_rate,
            'avg_sensitivity': np.mean(list(per_neuron_sensitivity.values())),
            'n_matched': n_matched,
            'n_unmatched': n_unmatched,
            'total_ref_spikes': total_ref_spikes,
            'total_comp_spikes': n_comp_spikes,
        }
        
        print(f"\n📊 压缩率 {ratio*100:.0f}% 结果:")
        print(f"  - 稳定性: {metrics['stability']*100:.2f}%")
        print(f"  - 丢失率: {metrics['dropout_rate']*100:.2f}%")
        print(f"  - 新颖率: {metrics['novelty_rate']*100:.2f}%")
        print(f"  - 平均灵敏度: {metrics['avg_sensitivity']*100:.2f}%")
        
        results.append({
            'keep_ratio': ratio,
            'metrics': metrics,
            'assignments': assignments,
            'confidences': confidences,
            'comp_rez': comp_rez,
        })
    
    return results


In [13]:
SAVE_PATH = Path('F:\\ACADEMIC') / '.test_data' / 'ZFM-02370_mini.imec0.ap.short.bin'
results_dir = SAVE_PATH.parent / 'kilosort4'

templates_ref = np.load(results_dir / 'templates.npy')      # (267, 61, 385)
clu_ref = np.load(results_dir / 'spike_clusters.npy')       # (n_spikes,)
st_ref = np.load(results_dir / 'spike_times.npy')           # (n_spikes,)
ops = np.load(results_dir / 'ops.npy', allow_pickle=True).item()

print("Reference data information")
print("="*60)
print(f"Number of reference templates: {templates_ref.shape[0]}")
print(f"Reference template shape: {templates_ref.shape}")
print(f"Total number of reference spikes: {len(st_ref)}")
print(f"Number of reference clusters: {len(np.unique(clu_ref))}")
print(f"Sampling rate: {ops.get('fs', 30000)} Hz")

# 计算每个参考神经元的尖峰数量
reference_counts = {}
for cluster_id in np.unique(clu_ref):
    reference_counts[cluster_id] = np.sum(clu_ref == cluster_id)

Reference data information
Number of reference templates: 267
Reference template shape: (267, 61, 383)
Total number of reference spikes: 137381
Number of reference clusters: 267
Sampling rate: 30000 Hz
